In [19]:
# Imports + basic paths for LogLite and Loghub
from pathlib import Path
from collections import deque
import struct


ROOT = Path().resolve()
LOGLITE = ROOT / "loglite" / "scripts"
LOGHUB = ROOT / "dataset" / "loghub"
linux_path = LOGHUB / "Linux" / "Linux_2k.log"

with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    total = sum(1 for _ in f)

# print(f"{ROOT}, {LOGLITE}, {LOGHUB}")

In [20]:
# Parse window.txt into an L-window structure

def load_l_window_from_txt(path: Path):

    """
    Parse a LogLite window dump (text format) into {length: [templates...]}.

    """

    window = {}
    current_len = None
    current_templates = []

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")
            if not line:
                continue
            if line.startswith("len="):
                # flush previous bucket
                if current_len is not None:
                    window[current_len] = current_templates
                current_len = int(line.split("=", 1)[1])
                current_templates = []
            elif line == "---":
                if current_len is not None:
                    window[current_len] = current_templates
                    current_len = None
                    current_templates = []
            else:
                if current_len is not None:
                    current_templates.append(line)

    if current_len is not None:
        window[current_len] = current_templates
    return window

l_window = load_l_window_from_txt(ROOT / "compressed_logs" / "Linux_2k.window.txt")

# print(l_window)


In [21]:
from collections import deque
import struct
from pathlib import Path

def keyword_search_loglite_binary(bin_path, parsed_l_window, query_keywords):
    # Constants from constants.h
    EACH_WINDOW_SIZE_COUNT = 3
    STREAM_ENCODER_COUNT = 13
    ORIGINAL_LENGTH_COUNT = 15
    MAX_LEN = 10000
    RLE_COUNT = 8
    EACH_WINDOW_SIZE = 1 << EACH_WINDOW_SIZE_COUNT

    # Normalize keywords
    keywords = [query_keywords] if isinstance(query_keywords, str) else list(query_keywords)
    results = []
    
    # Internal state for Window Synchronization
    window = {} # type: dict[int, deque[bytes]]

    # --- Bitstream Loading Logic ---
    data = Path(bin_path).read_bytes()
    if len(data) < 16: return results
    
    ulong_size = 8
    file_size = len(data)
    last_block_bits = struct.unpack('<Q', data[file_size - 8:])[0]
    blocks_bytes = data[:file_size - 8]
    num_blocks = len(blocks_bytes) // ulong_size
    blocks = struct.unpack('<' + 'Q' * num_blocks, blocks_bytes)
    bits_per_block = 64
    total_bits = (num_blocks - 1) * bits_per_block + (last_block_bits or bits_per_block)
    
    pos = 0

    def read_bit():
        nonlocal pos
        if pos >= total_bits: return None
        bit = (blocks[pos // 64] >> (pos % 64)) & 1
        pos += 1
        return bit

    def read_int(bit_count):
        v = 0
        for i in range(bit_count):
            b = read_bit()
            if b is not None and b: v |= (1 << i)
        return v

    # --- Processing Loop ---
    while True:
        flag = read_bit()
        if flag is None: break

        if flag == 0:  # TYPE 0: RAW LINE
            length = read_int(ORIGINAL_LENGTH_COUNT)
            line_bytes = bytearray(length)
            for i in range(length):
                line_bytes[i] = read_int(8)
            
            # Update Window (must store bytes for XOR operations)
            window[length] = deque([bytes(line_bytes)], maxlen=EACH_WINDOW_SIZE)
            
            line_str = line_bytes.decode('utf-8', 'ignore')
            if all(k in line_str for k in keywords):
                results.append(line_str)

        else:  # TYPE 1: COMPRESSED LINE
            window_id = read_int(EACH_WINDOW_SIZE_COUNT)
            rle_bit_len = read_int(STREAM_ENCODER_COUNT)
            
            # 1. DECODE RLE (Necessary to find length and update state)
            xor_delta = bytearray()
            bits_consumed = 0
            while bits_consumed < rle_bit_len:
                rle_type = read_bit()
                bits_consumed += 1
                if rle_type == 1: # Literal
                    xor_delta.append(read_int(8))
                    bits_consumed += 8
                else: # Zero Run
                    count = read_int(RLE_COUNT)
                    bits_consumed += RLE_COUNT
                    xor_delta.extend([0] * count)
            
            length = len(xor_delta)
            dq = window.get(length)
            
            if not dq or window_id >= len(dq):
                continue
            
            # 2. EFFICIENT QUERYING LOGIC
            template_bytes = dq[window_id]
            
            # Optimization: Check if keyword is in the Template (Static Part)
            # We convert keywords to bytes to stay in the "Compressed Domain" as long as possible
            is_match = False
            template_str = template_bytes.decode('utf-8', 'ignore')
            
            if all(k in template_str for k in keywords):
                is_match = True # Guaranteed match because it's in the static part
            
            # 3. RECONSTRUCTION (Always needed for Window Sync)
            # This mimics the SIMD blend logic
            reconstructed_bytes = bytearray(xor_delta)
            for i in range(length):
                if reconstructed_bytes[i] == 0:
                    reconstructed_bytes[i] = template_bytes[i]
            
            final_line_bytes = bytes(reconstructed_bytes)
            
            # Final check if we didn't match via template
            if not is_match:
                line_str = final_line_bytes.decode('utf-8', 'ignore')
                if all(k in line_str for k in keywords):
                    is_match = True
                    results.append(line_str)
            else:
                results.append(final_line_bytes.decode('utf-8', 'ignore'))

            # Keep decompressor state synchronized
            dq.append(final_line_bytes)

    return results

In [22]:
# Query 1: simple keyword search for word kernel (uncompressed vs compressed)

# Uncompressed baseline
with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    matches = [line.rstrip("\n") for line in f if "kernel" in line]

q1_matches_uncompressed = len(matches)
print("Query 1 - keyword 'kernel' (uncompressed):", q1_matches_uncompressed, "matches")

# Compressed-domain search over LogLite binary
bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary(bin_path, l_window, "kernel")
q1_matches_compressed = len(compressed_matches)
print("Query 1 - keyword 'kernel' (compressed):", q1_matches_compressed, "matches")

# Query 1: confusion matrix and metrics

actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")


Query 1 - keyword 'kernel' (uncompressed): 77 matches
Query 1 - keyword 'kernel' (compressed): 83 matches
TP: 77
FP: 6
FN: 0
TN: 1917
Accuracy:  0.9970
Precision: 0.9277
Recall:    1.0000
F1 score:  0.9625


In [23]:
# Query 2: variable value match for PID 28842


with linux_path.open("r", encoding="utf-8", errors="ignore")  as f:
    matches = [line.rstrip("\n") for line in f if "28842" in line]

q2_matches_uncompressed = len(matches)
print("Query 2 - PID 28842 (Uncompressed):", q2_matches_uncompressed, "matches")


bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary(bin_path, l_window, "28842")
q2_matches_compressed = len(compressed_matches)
print("Query 2 - PID 28842 (Compressed):", q2_matches_compressed, "matches")


# confusion matrix and metrics

actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")


Query 2 - PID 28842 (Uncompressed): 1 matches
Query 2 - PID 28842 (Compressed): 2 matches
TP: 1
FP: 1
FN: 0
TN: 1998
Accuracy:  0.9995
Precision: 0.5000
Recall:    1.0000
F1 score:  0.6667


In [24]:
# Query 3: Time stamp based filtering.

with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    matches = [line.rstrip("\n") for line in f if line.startswith("Jul  2")]

q3_matches_uncompressed = len(matches)
print("Query 3 - Timestamp 'Jul 2' (Uncompressed):", q3_matches_uncompressed, "matches")



bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary(bin_path, l_window, "Jul  2")
q3_matches_compressed = len(compressed_matches)
print("Query 3 - Timestamp 'Jul 2' (Compressed):", q3_matches_compressed, "matches")


actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")



Query 3 - Timestamp 'Jul 2' (Uncompressed): 41 matches
Query 3 - Timestamp 'Jul 2' (Compressed): 53 matches
TP: 41
FP: 12
FN: 0
TN: 1947
Accuracy:  0.9940
Precision: 0.7736
Recall:    1.0000
F1 score:  0.8723


In [25]:
# Query 4: substring search for failed

with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    matches = [line.rstrip("\n") for line in f if "failed" in line]
    
q4_matches_uncompressed = len(matches)
print("Query 4 - keyword 'failed' (Uncompressed):", q4_matches_uncompressed, "matches")


bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary(bin_path, l_window, "failed")
q4_matches_compressed = len(compressed_matches)
print("Query 4 - keyword 'failed' (Compressed):", q4_matches_compressed, "matches")



actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")



Query 4 - keyword 'failed' (Uncompressed): 47 matches
Query 4 - keyword 'failed' (Compressed): 48 matches
TP: 47
FP: 1
FN: 0
TN: 1952
Accuracy:  0.9995
Precision: 0.9792
Recall:    1.0000
F1 score:  0.9895


In [ ]:
# Query 5: multi-token serach of tokens not next to each other

with linux_path.open("r", encoding="utf-8", errors="ignore") as f:
    matches = []
    for raw in f:
        line = raw.rstrip("\n")
        if "sshd" in line and "failure" in line:
            matches.append(line)
            
q5_matches_uncompressed = len(matches)
print("Query 5 - 'sshd' and 'failure' (Uncompressed):", q5_matches_uncompressed, "matches")


bin_path = ROOT / "compressed_logs" / "Linux_2k.log.lite.b"
compressed_matches = keyword_search_loglite_binary(bin_path, l_window, ["sshd","failure"])
q5_matches_compressed = len(compressed_matches)
print("Query 5 - 'sshd' and 'failure' (Compressed):", q5_matches_compressed, "matches")


actual_positive = set(matches)
retrieved_positive = set(compressed_matches)

tp = len(actual_positive & retrieved_positive)
fp = len(retrieved_positive - actual_positive)
fn = len(actual_positive - retrieved_positive)
tn = total - tp - fp - fn

accuracy = (tp + tn) / total if total else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1_score:.4f}")



Query 5 - 'sshd' and 'failure' (Uncompressed): 489 matches
Query 5 - 'sshd' and 'failure' (Compressed): 494 matches
TP: 489
FP: 5
FN: 0
TN: 1506
Accuracy:  0.9975
Precision: 0.9899
Recall:    1.0000
F1 score:  0.9949
